In [1]:
import pandas as pd

In [2]:
sold = pd.read_csv("combined_sold.csv")
print(sold.shape)

/tmp/ipykernel_14232/4185666676.py:1: DtypeWarning: Columns (2,45,78,79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("combined_sold.csv")


(552253, 82)


In [3]:
# Restrict to residential sales before doing anything else. The raw data mixes
# property types with incompatible price scales under the same ClosePrice column:
# Residential (mean ~$1.15M) vs. ResidentialLease (mean ~$7.3K -- that's rent, not
# a sale price) vs. Land, CommercialSale, CommercialLease, ManufacturedInPark,
# BusinessOpportunity. Predicting all of these together with one regression target
# doesn't make sense, so scope down to Residential first.
print(f"PropertyType value counts before filtering:\n{sold['PropertyType'].value_counts(dropna=False)}\n")
sold = sold[sold['PropertyType'] == 'Residential'].copy()
sold.drop(columns=['PropertyType'], inplace=True)  # now constant -- no longer informative
print(f"sold shape after restricting to Residential: {sold.shape}")

# Drop rows with no target -- can't train or evaluate on these.
before = len(sold)
sold = sold[sold['ClosePrice'].notna()]
print(f"Dropped {before - len(sold)} rows with missing ClosePrice")

# Remove obvious data-entry errors in ClosePrice: sales under $1,000, and a
# price-per-sqft over $5,000 (99.9th percentile of price/sqft here is ~$3,600, so
# $5,000/sqft is a generous ceiling). This catches things like a $970M close price
# on a 1,976 sqft house -- a data error, not a real sale -- without touching the
# legitimate high end of the market.
has_area = sold['LivingArea'] > 0
ppsf = pd.Series(index=sold.index, dtype=float)
ppsf[has_area] = sold.loc[has_area, 'ClosePrice'] / sold.loc[has_area, 'LivingArea']
bad_price = (sold['ClosePrice'] < 1000) | (ppsf > 5000)
print(f"Dropping {bad_price.sum()} rows with implausible ClosePrice / price-per-sqft")
sold = sold[~bad_price].copy()

print(f"sold shape after outlier removal: {sold.shape}")

PropertyType value counts before filtering:
PropertyType
Residential            370878
ResidentialLease       125131
Land                    19790
ManufacturedInPark      15115
ResidentialIncome       14301
CommercialSale           3616
CommercialLease          3008
BusinessOpportunity       412
Resid                       1
NaN                         1
Name: count, dtype: int64

sold shape after restricting to Residential: (370878, 81)
Dropped 2 rows with missing ClosePrice
Dropping 117 rows with implausible ClosePrice / price-per-sqft
sold shape after outlier removal: (370759, 81)


### Robust Outlier Detection & Sanity Checks

Runs before any imputation/encoding stats are computed, so bad values never leak into group medians, county target-encoding, or the missing-indicator flags below. Two passes: (1) hard domain sanity checks -- logically impossible values, safe to drop outright -- and (2) a statistical outlier pass (Tukey's outer fence, 3x IQR) for values that are extreme but not impossible.

In [4]:
# --- Hard domain sanity checks ------------------------------------------
# Catch logically impossible values across the numeric features used later
# in modeling, beyond the ClosePrice check already done above (e.g. a $0
# list price, a negative bedroom count, a house "built" in the future).
# These are clear data-entry errors independent of any statistical
# distribution, so -- unlike the IQR pass below -- they're safe to drop
# outright. NaNs are left alone here (comparisons against NaN evaluate to
# False), since they're handled by the group-wise imputation further down,
# not by outlier removal.
current_year = pd.Timestamp.now().year

sanity_checks = {
    'OriginalListPrice <= 0': sold['OriginalListPrice'] <= 0,
    'LivingArea <= 0': sold['LivingArea'] <= 0,
    f'YearBuilt outside [1700, {current_year}]':
        sold['YearBuilt'].notna() & ~sold['YearBuilt'].between(1700, current_year),
    'BedroomsTotal < 0': sold['BedroomsTotal'] < 0,
    'BathroomsTotalInteger < 0': sold['BathroomsTotalInteger'] < 0,
    'Stories < 0': sold['Stories'] < 0,
    'GarageSpaces < 0': sold['GarageSpaces'] < 0,
    'MainLevelBedrooms < 0': sold['MainLevelBedrooms'] < 0,
    'LotSizeAcres < 0': sold['LotSizeAcres'] < 0,
}

bad_sanity = pd.Series(False, index=sold.index)
for label, mask in sanity_checks.items():
    print(f"{mask.sum():>6} rows fail: {label}")
    bad_sanity |= mask

print(f"\nDropping {bad_sanity.sum()} rows total (any check failed), out of {len(sold)}")
sold = sold[~bad_sanity].copy()
print(f"sold shape after hard sanity checks: {sold.shape}")

     1 rows fail: OriginalListPrice <= 0
   124 rows fail: LivingArea <= 0
     0 rows fail: YearBuilt outside [1700, 2026]
     0 rows fail: BedroomsTotal < 0
     0 rows fail: BathroomsTotalInteger < 0
     0 rows fail: Stories < 0
     0 rows fail: GarageSpaces < 0
     0 rows fail: MainLevelBedrooms < 0
     0 rows fail: LotSizeAcres < 0

Dropping 125 rows total (any check failed), out of 370759
sold shape after hard sanity checks: (370634, 81)


In [5]:
# --- Statistical outlier removal (Tukey's outer fence) -------------------
# For values that are extreme but not logically impossible (e.g. a
# legitimately huge, expensive house), flag anything outside 3x IQR from
# Q1/Q3 -- the "outer fence", more conservative than the usual 1.5x "inner
# fence" cutoff. Real-estate prices, lot sizes, etc. are naturally
# right-skewed, and a 1.5x cutoff would discard a lot of genuine high-end
# inventory rather than actual data errors. NaNs are ignored by
# quantile/comparison, same as the sanity checks above.
iqr_cols = [
    'ClosePrice', 'OriginalListPrice', 'LivingArea', 'BedroomsTotal',
    'BathroomsTotalInteger', 'Stories', 'GarageSpaces'
]

bad_stat = pd.Series(False, index=sold.index)
for col in iqr_cols:
    q1, q3 = sold[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 3 * iqr, q3 + 3 * iqr
    col_bad = (sold[col] < lower) | (sold[col] > upper)
    print(f"{col:>22}: fence = [{lower:,.2f}, {upper:,.2f}]  ->  {col_bad.sum()} rows flagged")
    bad_stat |= col_bad

print(f"\nDropping {bad_stat.sum()} rows total (flagged on any column), out of {len(sold)}")
sold = sold[~bad_stat].copy()
print(f"sold shape after statistical outlier removal: {sold.shape}")

            ClosePrice: fence = [-1,600,000.00, 3,475,000.00]  ->  11281 rows flagged
     OriginalListPrice: fence = [-1,569,000.00, 3,443,000.00]  ->  12363 rows flagged
            LivingArea: fence = [-1,646.00, 5,088.00]  ->  4234 rows flagged
         BedroomsTotal: fence = [0.00, 7.00]  ->  562 rows flagged
 BathroomsTotalInteger: fence = [-1.00, 6.00]  ->  2448 rows flagged
               Stories: fence = [-2.00, 5.00]  ->  0 rows flagged
          GarageSpaces: fence = [2.00, 2.00]  ->  133134 rows flagged

Dropping 140931 rows total (flagged on any column), out of 370634
sold shape after statistical outlier removal: (229703, 81)


In [6]:
 #Drop columns with a null rate of 50% or more
null_threshold = 0.5
         
for name, df in [('sold', sold)]:
    null_rate = df.isnull().mean()
    cols_to_drop = null_rate[null_rate >= null_threshold].index.tolist()
    print(f"{name}: dropping {len(cols_to_drop)} columns with >= {null_threshold:.0%} nulls -> {cols_to_drop}")
    df.drop(columns=cols_to_drop, inplace=True)
print(f"sold shape: {sold.shape}")

sold: dropping 25 columns with >= 50% nulls -> ['WaterfrontYN', 'BasementYN', 'CoListOfficeName', 'CoListAgentFirstName', 'CoListAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'ElementarySchool', 'BuilderName', 'SubdivisionName', 'BuyerAgencyCompensationType', 'BuyerAgencyCompensation', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'HighSchool', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']
sold shape: (229703, 56)


In [7]:
# Drop agent/office/board identifiers -- these are IDs, not property features.
# One-hot encoding them would add 190+ near-useless columns and risks leakage
# (a specific agent/office can proxy for a price range).
sold.drop(columns=['BuyerOfficeAOR', 'BuyerAgentAOR', 'ListAgentAOR'], inplace=True)

# StateOrProvince is redundant with the more granular CountyOrParish -- drop it.
sold.drop(columns=['StateOrProvince'], inplace=True)

# Levels is ordinal ("One"/"Two"/"ThreeOrMore") but stored as comma-joined combos
# (e.g. "One,Two,MultiSplit"). Convert to a numeric max-level plus a separate
# MultiSplit flag instead of one-hot encoding all 15 raw combos. Done here, before
# the missing-indicator/imputation cells below, so NumLevels' nulls get the same
# group-wise median treatment as the other numeric columns.
level_map = {'One': 1, 'Two': 2, 'ThreeOrMore': 3}
sold['Levels_MultiSplit'] = sold['Levels'].str.contains('MultiSplit', na=False).astype(int)
sold['NumLevels'] = sold['Levels'].apply(
    lambda v: max((level_map[p] for p in v.split(',') if p in level_map), default=None)
    if pd.notna(v) else None
)
sold.drop(columns=['Levels'], inplace=True)

print(f"sold shape after dropping ID columns and re-encoding Levels: {sold.shape}")


sold shape after dropping ID columns and re-encoding Levels: (229703, 53)


In [8]:
# Parse close date and pin down the test-month cutoff now, before any
# imputation/encoding statistics are computed below. Those statistics (group
# medians, county target-encoding) are restricted to pre-test-month rows further
# down, so the test set can't leak into features used anywhere in training.
sold['CloseDate'] = pd.to_datetime(sold['CloseDate'])
sold['CloseMonth'] = sold['CloseDate'].dt.to_period('M')
test_month = sold['CloseMonth'].max()
pre_test = sold['CloseMonth'] < test_month
print(f"Test month: {test_month} ({(~pre_test).sum()} rows) -- excluded from imputation/encoding stats below")

Test month: 2025-06 (7038 rows) -- excluded from imputation/encoding stats below


In [9]:
# Missing-indicator flags, created BEFORE imputation (must reflect original nulls)
numeric_cols = sold.select_dtypes(include='number').columns
cols_with_nulls = [c for c in numeric_cols if sold[c].isnull().any()]

for col in cols_with_nulls:
    sold[f'{col}_missing'] = sold[col].isnull().astype(int)

print(f"Added {len(cols_with_nulls)} missing-indicator columns: {cols_with_nulls}")

Added 16 missing-indicator columns: ['OriginalListPrice', 'Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'NumLevels']


In [10]:
# Group-wise median imputation with a fallback hierarchy:
# PropertySubType+ZIP -> PropertySubType+City -> PropertySubType+County -> PropertySubType -> global
# Each level only fills what the previous (more specific) level couldn't resolve.
# Medians are computed from pre-test-month rows only (see `pre_test` above) and
# broadcast to the whole dataset, so the test month never influences the values
# used to fill missing data -- including missing values within the test month
# itself.
group_hierarchy = [
    ['PropertySubType', 'PostalCode'],
    ['PropertySubType', 'City'],
    ['PropertySubType', 'CountyOrParish'],
    ['PropertySubType'],
    [],
]

for col in cols_with_nulls:
    for group_keys in group_hierarchy:
        train_only = sold[col].where(pre_test)
        if group_keys:
            group_medians = train_only.groupby([sold[k] for k in group_keys]).transform('median')
            sold[col] = sold[col].fillna(group_medians)
        else:
            sold[col] = sold[col].fillna(train_only.median())

remaining_nulls = sold[cols_with_nulls].isnull().sum().sum()
print(f"Remaining nulls in imputed columns after fallback chain: {remaining_nulls}")

Remaining nulls in imputed columns after fallback chain: 0


In [11]:
# CountyOrParish is meaningful but too high-cardinality (63 values) for one-hot --
# target-encode it as mean ClosePrice per county instead of adding 63 dummy columns.
# Computed from pre-test-month rows only (see `pre_test` above), so the test set's
# own close prices can't leak into this feature.
train_close = sold['ClosePrice'].where(pre_test)
county_means = train_close.groupby(sold['CountyOrParish']).transform('mean')
sold['CountyOrParish_MeanPrice'] = county_means.fillna(train_close.mean())
sold.drop(columns=['CountyOrParish'], inplace=True)

# Convert categorical (non-numeric) columns to numeric via one-hot encoding.
# Date/period columns and very high-cardinality columns need different treatment
# (date feature engineering / target-encoding, not one-hot), so they're skipped here.
categorical_cols = sold.select_dtypes(exclude='number').columns
date_cols = [c for c in categorical_cols if 'Date' in c or c == 'CloseMonth']

cardinality = sold[categorical_cols].nunique(dropna=True)
onehot_cols = [c for c in categorical_cols if c not in date_cols and 2 <= cardinality[c] <= 100]
high_card_cols = [c for c in categorical_cols if c not in date_cols and c not in onehot_cols]

print(f"One-hot encoding {len(onehot_cols)} columns: {onehot_cols}")
print(f"Skipping {len(date_cols)} date/period columns (need date feature engineering, not one-hot): {date_cols}")
print(f"Skipping {len(high_card_cols)} high-cardinality/ID-like columns (>100 unique values): {high_card_cols}")

sold = pd.get_dummies(sold, columns=onehot_cols, dummy_na=False, dtype=int)
print(f"sold shape after one-hot encoding: {sold.shape}")

One-hot encoding 8 columns: ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'PropertySubType', 'FireplaceYN', 'NewConstructionYN', 'latfilled', 'lonfilled']
Skipping 5 date/period columns (need date feature engineering, not one-hot): ['CloseDate', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'CloseMonth']
Skipping 17 high-cardinality/ID-like columns (>100 unique values): ['Flooring', 'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListOfficeName', 'BuyerOfficeName', 'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'MLSAreaMajor', 'MlsStatus', 'ListingId', 'City', 'HighSchoolDistrict', 'PostalCode']
sold shape after one-hot encoding: (229703, 93)


In [12]:
# Test set = most recent month of CloseDate (test_month, computed earlier before
# the imputation/encoding steps above). Train set = X months immediately
# preceding it, where X is tuned below rather than fixed.
monthly_counts = sold['CloseMonth'].value_counts().sort_index()
print("Rows per month:")
print(monthly_counts)

print(f"\nMost recent month (test set): {test_month}  ({monthly_counts[test_month]} rows)")
print("Note: months before 2023-05 have far fewer rows (partial data collection) -- "
      "keep this in mind if a large X reaches back that far.")
sold.to_csv("cleaned_sold.csv", index=False)
print(len(sold))

Rows per month:
CloseMonth
2022-01     1627
2022-02      906
2022-03      593
2022-04      290
2022-05      168
2022-06      103
2022-07       63
2022-08       65
2022-09       36
2022-10       21
2022-11       11
2022-12       16
2023-01       14
2023-02       12
2023-03       11
2023-04       66
2023-05     1532
2023-06     7302
2023-07     8307
2023-08     9936
2023-09     8688
2023-10     9005
2023-11     7980
2023-12     7820
2024-01     6921
2024-02     8118
2024-03     9790
2024-04    10609
2024-05    11416
2024-06    10344
2024-07    11093
2024-08    10502
2024-09     8979
2024-10    10060
2024-11     8850
2024-12     8845
2025-01     6689
2025-02     7536
2025-03     8968
2025-04     9776
2025-05     9597
2025-06     7038
Freq: M, Name: count, dtype: int64

Most recent month (test set): 2025-06  (7038 rows)
Note: months before 2023-05 have far fewer rows (partial data collection) -- keep this in mind if a large X reaches back that far.
229703


In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Tune X via a single-fold walk-forward validation: hold out the month right
# before the test month as a validation proxy, so the true test month is never
# touched while choosing X. This is a quick probe model just to rank candidate
# X values -- not the final model.
# Note: the imputation/target-encoding stats above are computed from all
# pre-test-month rows, which includes this validation month -- so there's a
# small amount of validation-month leakage left in this tuning step specifically
# (the final train/test split below is not affected). Not fixed here since it
# only affects the choice of X, not the final model's evaluation.
validation_month = test_month - 1

# Exclude the target; ID-like numeric columns that aren't real property features
# (ListingKeyNumeric is a DB sequence id that correlates with time, which would
# let the model cheat on the time-based split; StreetNumberNumeric is just the
# house number on the street); and DaysOnMarket, which isn't known until a
# listing closes or expires and so wouldn't be available at prediction time.
exclude_cols = ['ClosePrice', 'ListingKeyNumeric', 'StreetNumberNumeric', 'DaysOnMarket']
feature_cols = [c for c in sold.select_dtypes(include='number').columns if c not in exclude_cols]

val_df = sold[sold['CloseMonth'] == validation_month]

results = []
for X in range(1, 25):
    train_start = validation_month - X
    train_df = sold[(sold['CloseMonth'] >= train_start) & (sold['CloseMonth'] < validation_month)]
    if len(train_df) < len(feature_cols) * 2:
        continue
    model = LinearRegression()
    model.fit(train_df[feature_cols], train_df['ClosePrice'])
    preds = model.predict(val_df[feature_cols])
    mae = mean_absolute_error(val_df['ClosePrice'], preds)
    results.append((X, len(train_df), mae))

results_df = pd.DataFrame(results, columns=['X_months', 'train_rows', 'val_mae'])
print(results_df.sort_values('val_mae'))

best_X = int(results_df.loc[results_df['val_mae'].idxmin(), 'X_months'])
print(f"\nBest X by validation MAE: {best_X} months")

    X_months  train_rows       val_mae
23        24      209066  41993.498740
22        23      207534  42008.350780
21        22      200232  42074.503865
20        21      191925  42203.901237
19        20      181989  42398.611540
18        19      173301  42546.761949
3          4       32969  43403.458424
0          1        9776  43450.079785
17        18      164296  43542.914546
16        17      156316  43689.558229
15        16      148496  43788.648763
14        15      141575  43814.922260
13        14      133457  43873.319643
12        13      123667  43968.451647
2          3       26280  44202.003037
11        12      113058  44253.232589
1          2       18744  44565.150890
10        11      101642  45181.991108
9         10       91298  45319.110946
8          9       80205  45918.668090
7          8       69703  46707.414070
6          7       60724  47417.781002
5          6       50664  47763.873196
4          5       41814  47847.792640

Best X by validation MAE

In [14]:
# Final split: test = most recent month, train = best_X months immediately before it
train_start = test_month - best_X
train_mask = (sold['CloseMonth'] >= train_start) & (sold['CloseMonth'] < test_month)
test_mask = sold['CloseMonth'] == test_month

train_set = sold[train_mask].copy()
test_set = sold[test_mask].copy()

print(f"Train: {train_start} to {test_month - 1}  ({len(train_set)} rows)")
print(f"Test:  {test_month}  ({len(test_set)} rows)")

Train: 2023-06 to 2025-05  (217131 rows)
Test:  2025-06  (7038 rows)


In [39]:
test_set.shape
test_set.to_csv("basic_test_set.csv")

In [40]:
train_set.shape
train_set.to_csv("basic_train_set.csv")

In [17]:
# Say 'Target_Variable' is the name of the column you want to predict
y_train = train_set['ClosePrice']
y_test = test_set['ClosePrice']

In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


# EDA on Features

#### Feature Testing: Bathrooms

In [19]:

X_train = train_set[['BathroomsTotalInteger']]
X_test = test_set[['BathroomsTotalInteger']]

# Initialize the model
model = LinearRegression()

# Train the model (fitting the line to the data)
model.fit(X_train, y_train)

# Predict the target values for the test set
y_pred = model.predict(X_test)

In [20]:


# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Mean Squared Error: 301497960684.41
R-squared Score: 0.12


#### Feature Testing: OriginalListPrice

In [21]:

X_train = train_set[['OriginalListPrice']]
X_test = test_set[['OriginalListPrice']]

# Initialize the model
model = LinearRegression()

# Train the model (fitting the line to the data)
model.fit(X_train, y_train)

# Predict the target values for the test set
y_pred = model.predict(X_test)

In [22]:


# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Mean Squared Error: 9035916776.93
R-squared Score: 0.97


#### Feature Testing: BedroomsTotal

In [23]:

X_train = train_set[['BedroomsTotal']]
X_test = test_set[['BedroomsTotal']]

# Initialize the model
model = LinearRegression()

# Train the model (fitting the line to the data)
model.fit(X_train, y_train)

# Predict the target values for the test set
y_pred = model.predict(X_test)

In [24]:


# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Mean Squared Error: 317837505766.76
R-squared Score: 0.08


In [25]:

X_train = train_set[['LotSizeAcres']]
X_test = test_set[['LotSizeAcres']]

# Initialize the model
model = LinearRegression()

# Train the model (fitting the line to the data)
model.fit(X_train, y_train)

# Predict the target values for the test set
y_pred = model.predict(X_test)

In [26]:


# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Mean Squared Error: 346802042538.71
R-squared Score: -0.01


#### Feature Testing: Stories

In [27]:

X_train = train_set[['Stories']]
X_test = test_set[['Stories']]

# Initialize the model
model = LinearRegression()

# Train the model (fitting the line to the data)
model.fit(X_train, y_train)

# Predict the target values for the test set
y_pred = model.predict(X_test)

In [28]:


# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Mean Squared Error: 343198890922.37
R-squared Score: 0.00


#### Feature Testing: MainLevelBedrooms

In [29]:

X_train = train_set[['MainLevelBedrooms']]
X_test = test_set[['MainLevelBedrooms']]

# Initialize the model
model = LinearRegression()

# Train the model (fitting the line to the data)
model.fit(X_train, y_train)

# Predict the target values for the test set
y_pred = model.predict(X_test)

In [30]:


# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Mean Squared Error: 346692004018.22
R-squared Score: -0.01


#### Feature Testing: GarageSpaces

In [31]:

X_train = train_set[['GarageSpaces']]
X_test = test_set[['GarageSpaces']]

# Initialize the model
model = LinearRegression()

# Train the model (fitting the line to the data)
model.fit(X_train, y_train)

# Predict the target values for the test set
y_pred = model.predict(X_test)

In [32]:


# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Mean Squared Error: 346816154632.40
R-squared Score: -0.01


#### (all feature_cols)

In [33]:

X_train_full = train_set[feature_cols]
X_test_full = test_set[feature_cols]

model_full = LinearRegression()
model_full.fit(X_train_full, y_train)
y_pred_full = model_full.predict(X_test_full)

mse_full = mean_squared_error(y_test, y_pred_full)
r2_full = r2_score(y_test, y_pred_full)

print(f"Mean Squared Error: {mse_full:.2f}")
print(f"R-squared Score: {r2_full:.2f}")

Mean Squared Error: 6259092246.62
R-squared Score: 0.98


In [34]:

exclude_cols_minus_price = ['ClosePrice', 'ListingKeyNumeric', 'StreetNumberNumeric', 'DaysOnMarket', 'OriginalListPrice', 'ListPrice']
feature_cols_minus_price = [c for c in sold.select_dtypes(include='number').columns if c not in exclude_cols_minus_price]
print(feature_cols_minus_price)

X_train_full = train_set[feature_cols_minus_price]
X_test_full = test_set[feature_cols_minus_price]



model_full = LinearRegression()
model_full.fit(X_train_full, y_train)
y_pred_full = model_full.predict(X_test_full)

mse_full = mean_squared_error(y_test, y_pred_full)
r2_full = r2_score(y_test, y_pred_full)

print(f"Mean Squared Error: {mse_full:.2f}")
print(f"R-squared Score: {r2_full:.2f}")

['ListingKey', 'Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger', 'BedroomsTotal', 'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'Levels_MultiSplit', 'NumLevels', 'OriginalListPrice_missing', 'Latitude_missing', 'Longitude_missing', 'LivingArea_missing', 'ParkingTotal_missing', 'LotSizeAcres_missing', 'YearBuilt_missing', 'StreetNumberNumeric_missing', 'BathroomsTotalInteger_missing', 'Stories_missing', 'LotSizeArea_missing', 'MainLevelBedrooms_missing', 'GarageSpaces_missing', 'AssociationFee_missing', 'LotSizeSquareFeet_missing', 'NumLevels_missing', 'CountyOrParish_MeanPrice', 'ViewYN_False', 'ViewYN_True', 'PoolPrivateYN_False', 'PoolPrivateYN_True', 'AttachedGarageYN_False', 'AttachedGarageYN_True', 'PropertySubType_Cabin', 'PropertySubType_CoOwnership', 'PropertySubType_Condominium', 'PropertySubType_Duplex', 'PropertySubType_Farm', 'PropertySubType_Loft', 'PropertySu

Mean Squared Error: 131079443658.05
R-squared Score: 0.62


In [35]:
feature_cols = ['Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger', 'BedroomsTotal', 'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'Levels_MultiSplit', 'NumLevels']
X_train_full = train_set[feature_cols]
X_test_full = test_set[feature_cols]



model_full = LinearRegression()
model_full.fit(X_train_full, y_train)
y_pred_full = model_full.predict(X_test_full)

mse_full = mean_squared_error(y_test, y_pred_full)
r2_full = r2_score(y_test, y_pred_full)

print(f"Mean Squared Error: {mse_full:.2f}")
print(f"R-squared Score: {r2_full:.2f}")

Mean Squared Error: 211082375822.03
R-squared Score: 0.39


#### OriginalListPrice Outlier Check

In [36]:
print(sold['OriginalListPrice'].describe())

ratio = sold['ClosePrice'] / sold['OriginalListPrice']
print("\nClosePrice / OriginalListPrice ratio:")
print(ratio.describe())

extreme = sold[(ratio > 3) | (ratio < 1 / 3)]
print(f"\n{len(extreme)} rows where ClosePrice is >3x or <1/3 of OriginalListPrice")
extreme[['OriginalListPrice', 'ClosePrice']].sort_values('OriginalListPrice', ascending=False).head(10)

count    2.297030e+05
mean     1.004693e+06
std      5.689666e+05
min      1.000000e+00
25%      6.123175e+05
50%      8.490000e+05
75%      1.250000e+06
max      3.440000e+06
Name: OriginalListPrice, dtype: float64

ClosePrice / OriginalListPrice ratio:
count    2.297030e+05
mean     3.087813e+01
std      5.828150e+03
min      3.768662e-02
25%      9.692315e-01
50%      1.000000e+00
75%      1.033333e+00
max      1.800000e+06
dtype: float64

344 rows where ClosePrice is >3x or <1/3 of OriginalListPrice


,OriginalListPrice,ClosePrice
94640,3324900.0,300000.0
318262,3119900.0,309900.0
440151,2295000.0,235500.0
138008,1699000.0,155000.0
7696,1490000.0,256404.0
510143,1200000.0,100000.0
394084,1199900.0,122500.0
406895,1121990.0,104090.0
414415,1099900.0,112000.0
377562,1098000.0,110000.0


#### OriginalListPrice Cleanup & Re-test

In [37]:
has_area = sold['LivingArea'] > 0
list_ppsf = pd.Series(index=sold.index, dtype=float)
list_ppsf[has_area] = sold.loc[has_area, 'OriginalListPrice'] / sold.loc[has_area, 'LivingArea']

bad_list_price = (sold['OriginalListPrice'] <= 0) | (list_ppsf > 5000)
print(f"{bad_list_price.sum()} rows with implausible OriginalListPrice "
      f"(<=0 or >$5,000/sqft), out of {len(sold)}")
sold.loc[bad_list_price, ['OriginalListPrice', 'ClosePrice', 'LivingArea']].describe()

0 rows with implausible OriginalListPrice (<=0 or >$5,000/sqft), out of 229703


,OriginalListPrice,ClosePrice,LivingArea
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
25%,NaN,NaN,NaN
50%,NaN,NaN,NaN
75%,NaN,NaN,NaN
max,NaN,NaN,NaN


In [38]:
def bad_original_list_price(df):
    has_area = df['LivingArea'] > 0
    ppsf = pd.Series(index=df.index, dtype=float)
    ppsf[has_area] = df.loc[has_area, 'OriginalListPrice'] / df.loc[has_area, 'LivingArea']
    return (df['OriginalListPrice'] <= 0) | (ppsf > 5000)

train_bad = bad_original_list_price(train_set)
test_bad = bad_original_list_price(test_set)
print(f"Dropping {train_bad.sum()} bad rows from train, {test_bad.sum()} from test")

train_clean = train_set[~train_bad]
test_clean = test_set[~test_bad]
y_train_clean = y_train[~train_bad]
y_test_clean = y_test[~test_bad]

X_train = train_clean[['OriginalListPrice']]
X_test = test_clean[['OriginalListPrice']]

model = LinearRegression()
model.fit(X_train, y_train_clean)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test_clean, y_pred)
r2 = r2_score(y_test_clean, y_pred)
print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Dropping 0 bad rows from train, 0 from test
Mean Squared Error: 9035916776.93
R-squared Score: 0.97


Next Steps: Split by County Price Averages.